In [335]:
import pandas as pd
import plotly.express as px

from tabulate import tabulate
from termcolor import colored
#from babel.numbers import format_decimal

# Dash Boards
from plotly.subplots import make_subplots
#import plotly.graph_objects as go

df_business=pd.read_csv('./dataset/Business Unit.csv')
df_customer=pd.read_csv('./dataset/Customer.csv')
df_dates=pd.read_csv('./dataset/Dates.csv')
df_financial=pd.read_excel('./dataset/Financial Sample.xlsx')

# Fix
df_financial.rename(columns={' Sales': 'Sales'}, inplace=True)
df_financial['Discount Band']=df_financial['Discount Band'].fillna('None')

# Cria os Trimestres em Financials
df_financial['Dates_Quarter'] = df_financial['Month Number'].map({
    1: 'Q1', 2: 'Q1', 3: 'Q1',
    4: 'Q2', 5: 'Q2', 6: 'Q2',
    7: 'Q3', 8: 'Q3', 9: 'Q3',
    10: 'Q4', 11: 'Q4', 12: 'Q4'
})


In [336]:

def dataset_info():
      def print_dataframe_info(df, name):
            print(colored(str(f"{name}"),'green')+" contem " +
                  colored(str(f"{df.shape[0]}"),'red')+" linhas e "+
                  colored(str(f"{df.shape[1]}"),'red')+" colunas")
            print(colored(str("Sobre os Data Sets:\n"),'yellow'))
            print(tabulate(df.head(3), headers='keys', tablefmt='pretty')+"\n")

      print_dataframe_info(df_business, "Business")
      print_dataframe_info(df_customer, "Customer")
      print_dataframe_info(df_dates, "Dates")
      print_dataframe_info(df_financial, "Financial")

def financial_details():
      print(colored("Valores em Finacial:\n","light_magenta"))

      print('Discounts '+
            colored("["+str(f"{df_financial['Discounts'].min():.2f}, {df_financial['Discounts'].mean():.2f}, {df_financial['Discounts'].max():.2f}"),"green")+"]")
      print('Discount Band '+
            colored(str(f"[{', '.join(str(val) for val in df_financial['Discount Band'].unique())}]"),"green"))
      print('Product '+ 
            colored(str(f"[{', '.join(str(val) for val in df_financial['Product'].unique())}]"),"green"))
      print('Segment '+ 
            colored(str(f"[{', '.join(str(val) for val in df_financial['Segment'].unique())}]"),"green"))
      print('Country '+ 
            colored(str(f"[{', '.join(str(val) for val in df_financial['Country'].unique())}]"),"green"))

def columns_info():
      def print_column_types(df, name):
            column_types = pd.DataFrame(df.dtypes, columns=['Type'])
            column_types.index.name = 'Column Name'
            print(colored(f"\t\t{name}",'green'))
            print(tabulate(column_types, headers='keys', tablefmt='pretty')+"\n")
            
      print_column_types(df_business, "Business")
      print_column_types(df_customer, "Customer")
      print_column_types(df_dates, "Dates")
      print_column_types(df_financial, "Financial")

In [337]:
# Generale means "No Segmented"
# Global means "No Localized"

In [358]:
# Soma de Vendas por Produto Generale Global
df_sum_product_sales = (
    df_financial.groupby('Product')['Sales'].sum().sort_values(ascending=False).reset_index()
    .assign(formatted_sales= lambda x: x['Sales'].apply(lambda x: f"{(x / 1_000_000):.2f} Mi"))
)

fig_sum_product_sales = px.pie(df_sum_product_sales, 
             names='Product',      
             values='Sales',          
             title='Soma de Vendas por Produto - Global',
             hole=0,
             ).update_traces(text=df_sum_product_sales['formatted_sales'],
                             textinfo='percent+text', textposition='inside').update_layout(width=600,height=400)#.show()

In [339]:
# Média de Sale Price por Product Generale Global
df_media_sale_price = df_financial.groupby('Product')['Sale Price'].mean().sort_values(ascending=False).reset_index() 

fig_media_sale_price = px.line(df_media_sale_price, 
              x='Product',        
              y='Sale Price',     
              title='Média de Sale Price por Product - Global',  
              markers=True
              ).update_layout(width=600,height=400) #.show()


In [340]:
#Soma das Vendas por Trimestre e Segmento Global
df_sales_tri_segment = df_financial.groupby(['Segment', 'Dates_Quarter'])['Sales'].sum().sort_values(ascending=False).reset_index()

fig_sales_tri_segment = px.bar(
    df_sales_tri_segment,  
    x='Dates_Quarter',   
    y='Sales',           
    color='Segment',      
    barmode='group',     
    title='Soma das Vendas por Trimestre e Segmento',
    category_orders={'Dates_Quarter': ['Q1', 'Q2', 'Q3', 'Q4']}
).update_layout(width=600,height=400)#.show()

In [341]:
# Soma de vendas e Soma de Unidades Vendidas
print("Total Sales: "+f"{df_financial['Sales'].sum() / 1_000_000:.2f} Mi")
print("Units Sold: " + f"{df_financial['Units Sold'].sum() / 1_000_000:.2f} Mi")

Total Sales: 118.73 Mi
Units Sold: 1.13 Mi


In [342]:
# Soma de Profit por Country Genrale
df_sum_profit_country = (
    df_financial.groupby('Country')['Profit'].sum().sort_values(ascending=False).reset_index()
    .assign(formatted_sum_profit = lambda x: x['Profit'].apply(lambda x: f"{(x / 1_000_000):.2f} Mi"))
)
fig_sum_profit_country = px.pie(df_sum_profit_country, 
             names='Country',      
             values='Profit',          
             title='Soma de Profit por Country',
             hole=0).update_traces(
   text=df_sum_profit_country['formatted_sum_profit'],
   textinfo='percent+text', textposition='inside').update_layout(width=600,height=400) #.show()

In [343]:
# Média de profit por Ano Mês Generale Global
df_mean_profit_month = (
    df_financial.groupby(['Year','Month Name', 'Month Number'])['Profit'].sum().reset_index()
    .assign(month_year=lambda x: x['Month Name'] + ' ' + x['Year'].astype(str))        #Coluna para o Agrupamento
    .sort_values(by=['Year','Month Number']).reset_index(drop=True)
)
fig_mean_profit_month = px.histogram(df_mean_profit_month, 
             x='month_year',      
             y='Profit',          
             title='Soma de Lucro por Mês/Ano').update_layout(width=600,height=400) #.show()

In [344]:
#Soma da Sales por Country Generale
df_sum_sales_country = (
    df_financial.groupby(['Country'])['Sales'].sum().reset_index()
)
fig_sum_sales_country = px.bar(df_sum_sales_country, 
             x='Country',      
             y='Sales',       
             title='Soma de vendas por País').update_layout(width=800,height=400)#.show()

In [345]:
df_sum_profit_segment = (
    df_financial.groupby('Segment').sum('Profit').reset_index()
    .assign(formated_profit_sum=lambda x: x['Profit'].apply(lambda x: f"{(x / 1_000_000):.2f} Mi")).reset_index(drop=True)
)

fig_sum_profit_segment = px.pie(df_sum_profit_segment, 
             names='Segment',      
             values='Profit',          
             title='Soma de Profit por Country',
             hole=0).update_traces(
   text=df_sum_profit_segment['formated_profit_sum'],
   textinfo='percent+text', textposition='inside').update_layout(width=600,height=400) #.show()


In [346]:
df_sum_product_country_unit = df_financial.groupby(['Product', 'Country'])['Units Sold'].sum().sort_values(ascending=False).reset_index()

fig_sum_product_country_unit = px.scatter_geo(
    df_sum_product_country_unit,
    locations='Country',  
    locationmode='country names', 
    size='Units Sold',  
    color='Product', 
    hover_name='Country', 
    title='Mapa de Vendas Produto por País'
).update_layout(width=800,height=600) #.show()

In [347]:
# Valores Negativos em SUMProfit indicam Prejuízo  LOGO .abs() é ilógico
#OU os dados para um segmento estão errados
#print(df_financial.groupby(['Segment', 'Country'])['Profit'].sum())
#Enterprise        Canada                      -121508.750
#                  France                       -95749.375
#                  Germany                     -101473.750
#                  Mexico                      -120678.750
#                  United States of America    -175135.000

df_sum_profit_segment_country = df_financial.groupby(['Segment', 'Country'])['Profit'].sum().abs().reset_index()

fig_sum_profit_segment_country = px.scatter_geo(
    df_sum_profit_segment_country,
    locations='Country',  
    locationmode='country names', 
    size='Profit',  
    color='Segment', 
    hover_name='Country', 
    title='Mapa de Lucro por Segment/País'
).update_layout(width=800,height=600) #.show()

In [348]:
#dataset_info()
#financial_details()
#columns_info()

In [362]:
#Faltam alguns Controles, Country, Ano_Trimestre

fig_relatorio_vendas_segmento_tri = make_subplots(
    rows=2, cols=2,                  
    subplot_titles=('Soma de Sales por Product', 
                    'Média de Sales Proice por Product', 
                    'Soma de Sales por Ano, Trimestre e Segmento'),
    specs=[[{'type': 'pie'}, {'type': 'histogram'}],
        [{'type': 'bar','colspan':2}, None]]
)

for trace in fig_sum_product_sales.data:
    fig_relatorio_vendas_segmento_tri.add_trace(trace, row=1, col=1)

for trace in fig_media_sale_price.data:
    fig_relatorio_vendas_segmento_tri.add_trace(trace, row=1, col=2)

for trace in fig_sales_tri_segment.data:
    fig_relatorio_vendas_segmento_tri.add_trace(trace, row=2, col=1)

fig_relatorio_vendas_segmento_tri.update_layout(
    title_text="Relátorio de Vendas por Segmento e Trimestre",
    showlegend=True,
    width=1400,  
    height=800
).show()

fig_relatorio_vendas_segmento_tri.write_image("fig/fig_pag1.jpg", format="jpg")


ValueError: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido


In [ ]:
# Relatório de Vendas por País e Lucro
fig_relatorio_vendas_segmento_tri = make_subplots(
    rows=2, cols=2,                  
    subplot_titles=('Soma de Profit por Country', 
                    'Média de Profit por Ano/Mês', 
                    'Soma de Sales por Country'),
    specs=[[{'type': 'pie'}, {'type': 'histogram'}],
        [{'type': 'bar', 'colspan':2}, None]]
)

# Adicionando os gráficos aos subgráficos
for trace in fig_sum_profit_country.data:
    fig_relatorio_vendas_segmento_tri.add_trace(trace, row=1, col=1)

for trace in fig_mean_profit_month.data:
    fig_relatorio_vendas_segmento_tri.add_trace(trace, row=1, col=2)

for trace in fig_sum_sales_country.data:
    fig_relatorio_vendas_segmento_tri.add_trace(trace, row=2, col=1)

# Atualizando o layout
fig_relatorio_vendas_segmento_tri.update_layout(
    title_text="Relatório de Vendas e Lucros",
    showlegend=True,
    width=1200,  
    height=800
).show()

fig_relatorio_vendas_segmento_tri.write_image("fig/fig_pag2.jpg", format="jpg")


In [ ]:
# Relatório Distruição de Lucros, vendas e unidades por país e Segmento
fig_relatorio_profits_sales_country_segment = make_subplots(
    rows=2, cols=2,      
    subplot_titles=('Soma de Profit por Segment', 
                    'Soma de Units Sales, Segment e Country', 
                    'Soma de Profit Segment e Country'),
    specs=[[{'type': 'pie'}, {'type': 'scattergeo'}],[{'type': 'scattergeo', 'colspan':2},None]]
)

# Adicionando os gráficos aos subgráficos
for trace in fig_sum_profit_segment.data:
    fig_relatorio_profits_sales_country_segment.add_trace(trace, row=1, col=1)

for trace in fig_sum_product_country_unit.data:
    fig_relatorio_profits_sales_country_segment.add_trace(trace, row=1, col=2)

for trace in fig_sum_profit_segment_country.data:
    fig_relatorio_profits_sales_country_segment.add_trace(trace, row=2, col=1)

# Atualizando o layout
fig_relatorio_profits_sales_country_segment.update_layout(
    title_text="Distruição de Lucros, vendas e unidades por país e Segmento",
    showlegend=True,
    width=1400,  
    height=800
).show()

fig_relatorio_profits_sales_country_segment.write_image("fig/fig_pag3.jpg", format="jpg")